In [2]:
# Data manipulation
import pandas as pd
import numpy as np

# Text preprocessing
import re
import string

# File handling
from pathlib import Path

# NLTK
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Progress bar
from tqdm import tqdm

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Download required NLTK resources (only first time)
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

# Enable progress_apply()
tqdm.pandas()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sarve\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sarve\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\sarve\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [3]:
# Define dataset paths
DATA_DIR = Path("../data/raw")

FAKE_PATH = DATA_DIR / "Fake.csv"
REAL_PATH = DATA_DIR / "Real.csv"

# Load datasets
fake_df = pd.read_csv(FAKE_PATH)
real_df = pd.read_csv(REAL_PATH)

# Display basic information
print("=" * 50)
print("Datasets Loaded Successfully")
print("=" * 50)
print(f"Fake News Dataset Shape : {fake_df.shape}")
print(f"Real News Dataset Shape : {real_df.shape}")

Datasets Loaded Successfully
Fake News Dataset Shape : (23481, 4)
Real News Dataset Shape : (21417, 4)


In [4]:
# Assign labels
fake_df["label"] = 1      # Fake News
real_df["label"] = 0      # Real News

# Merge datasets
df = pd.concat([fake_df, real_df], ignore_index=True)

# Shuffle dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Display information
print("=" * 50)
print("Merged Dataset")
print("=" * 50)
print(f"Dataset Shape : {df.shape}")

print("\nClass Distribution:")
print(df["label"].value_counts())

Merged Dataset
Dataset Shape : (44898, 5)

Class Distribution:
label
1    23481
0    21417
Name: count, dtype: int64


In [5]:
# Drop unnecessary columns
df.drop(columns=["subject", "date"], inplace=True)

print("Remaining Columns:")
print(df.columns.tolist())

print("\nDataset Shape:")
print(df.shape)

Remaining Columns:
['title', 'text', 'label']

Dataset Shape:
(44898, 3)


In [6]:
print(f"Dataset Shape Before Removing Duplicates : {df.shape}")

# Remove duplicates based on title and text
df.drop_duplicates(subset=["title", "text"], inplace=True)

# Reset index
df.reset_index(drop=True, inplace=True)

print(f"Dataset Shape After Removing Duplicates : {df.shape}")

Dataset Shape Before Removing Duplicates : (44898, 3)
Dataset Shape After Removing Duplicates : (39105, 3)


In [7]:
# Remove rows with empty text

df["text"] = df["text"].fillna("").astype(str)

empty_rows = (df["text"].str.strip() == "").sum()
print(f"Empty text rows: {empty_rows}")

df = df[df["text"].str.strip() != ""].reset_index(drop=True)

print(f"Dataset shape after removing empty text rows: {df.shape}")

Empty text rows: 447
Dataset shape after removing empty text rows: (38658, 3)


In [8]:
# Combine title and text

df["content"] = df["title"].fillna("") + " " + df["text"].fillna("")

df["content"] = df["content"].str.strip()

df[["title", "content"]].head()

,title,content
0,Ben Stein Calls Out 9th Circuit Court: Committ...,Ben Stein Calls Out 9th Circuit Court: Committ...
1,Trump drops Steve Bannon from National Securit...,Trump drops Steve Bannon from National Securit...
2,Puerto Rico expects U.S. to lift Jones Act shi...,Puerto Rico expects U.S. to lift Jones Act shi...
3,OOPS: Trump Just Accidentally Confirmed He Le...,OOPS: Trump Just Accidentally Confirmed He Lea...
4,Donald Trump heads for Scotland to reopen a go...,Donald Trump heads for Scotland to reopen a go...


In [9]:
df = df[["content", "label"]]

df.head()

,content,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,1
1,Trump drops Steve Bannon from National Securit...,0
2,Puerto Rico expects U.S. to lift Jones Act shi...,0
3,OOPS: Trump Just Accidentally Confirmed He Lea...,1
4,Donald Trump heads for Scotland to reopen a go...,0


### **Summary**

So far, we have merged the Fake and Real news datasets into a single dataframe, removed unnecessary columns (`subject` and `date`), eliminated duplicate and empty-text records, and combined the `title` and `text` columns into a single `content` column. The dataset is now clean and ready for text preprocessing before feature extraction.

### Text Preprocessing

In this step, we define a preprocessing function to clean the news articles. The function performs basic text cleaning, including converting text to lowercase, removing URLs, HTML tags, punctuation, numbers, stopwords, and extra spaces, followed by lemmatization. This helps reduce noise and prepares the text for feature extraction using TF-IDF.

In [10]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))


def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenize
    words = text.split()

    # Remove stopwords and lemmatize
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

### Apply Text Preprocessing

After defining the preprocessing function, we apply it to the `content` column. This cleans every news article using the same preprocessing steps and creates a consistent text format for feature extraction.

In [11]:
# Apply preprocessing to the content column

df["content"] = df["content"].progress_apply(preprocess_text)

100%|██████████| 38658/38658 [00:50<00:00, 765.24it/s]


In [12]:
# Compare original and cleaned text

for i in range(3):
    print(f"\nArticle {i+1}")
    print("-" * 60)
    print("Original:")
    print(fake_df.loc[i, "title"][:150])
    print()
    print("Processed:")
    print(df.loc[i, "content"][:150])


Article 1
------------------------------------------------------------
Original:
 Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing

Processed:
ben stein call th circuit court committed ‘coup d’état’ constitution st century wire say ben stein reputable professor pepperdine university also holl

Article 2
------------------------------------------------------------
Original:
 Drunk Bragging Trump Staffer Started Russian Collusion Investigation

Processed:
trump drop steve bannon national security council washington reuters u president donald trump removed chief strategist steve bannon national security 

Article 3
------------------------------------------------------------
Original:
 Sheriff David Clarke Becomes An Internet Joke For Threatening To Poke People ‘In The Eye’

Processed:
puerto rico expects u lift jones act shipping restriction reuters puerto rico governor ricardo rossello said wednesday expected federal government wai


In [13]:
df.head()

,content,label
0,ben stein call th circuit court committed ‘cou...,1
1,trump drop steve bannon national security coun...,0
2,puerto rico expects u lift jones act shipping ...,0
3,oops trump accidentally confirmed leaked israe...,1
4,donald trump head scotland reopen golf resort ...,0


In [14]:
# Final cleanup before saving

df = df.dropna(subset=["content"])

df["content"] = df["content"].astype(str)

df = df[df["content"].str.strip() != ""]

df.reset_index(drop=True, inplace=True)

print("Final dataset shape:", df.shape)
print(df.isnull().sum())

Final dataset shape: (38653, 2)
content    0
label      0
dtype: int64


In [15]:
output_path = Path("../data/processed/processed_news.csv")

df.to_csv(output_path, index=False)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!
